<a href="https://colab.research.google.com/github/sagar0226/100_days_machine_learing/blob/MachineLearningProject/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

#libraries needed for tensorflow processing
import tensorflow as tf
import numpy as np
import random
import json

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [17]:
#load the intents.json file from our local devices
from google.colab import files
files.upload()


Saving intents.json to intents (2).json


{'intents (2).json': b'{\r\n        "intents": [\r\n                {\r\n                        "tag": "greeting",\r\n                        "patterns": [\r\n                                "Hi",\r\n                                "How are you",\r\n                                "Is anyone there?",\r\n                                "Hello",\r\n                                "Good day"\r\n                        ],\r\n                        "responses": [\r\n                                "Hello, thanks for visiting",\r\n                                "Good to see you again",\r\n                                "Hi there, how can I help?"\r\n                        ],\r\n                        "context_set": ""\r\n                },\r\n                {\r\n                        "tag": "goodbye",\r\n                        "patterns": [\r\n                                "Bye",\r\n                                "See you later",\r\n                                "Goodbye"\r\n 

In [14]:
# Load the chatbot intent files, ensuring the 'intents' variable holds the full JSON structure.
with open('intents.json') as json_data:
  intents=json.load(json_data)

In [18]:
intents

{'intents': [{'tag': 'greeting',
   'patterns': ['Hi', 'How are you', 'Is anyone there?', 'Hello', 'Good day'],
   'responses': ['Hello, thanks for visiting',
    'Good to see you again',
    'Hi there, how can I help?'],
   'context_set': ''},
  {'tag': 'goodbye',
   'patterns': ['Bye', 'See you later', 'Goodbye'],
   'responses': ['See you later, thanks for visiting',
    'Have a nice day',
    'Bye! Come back again soon.']},
  {'tag': 'thanks',
   'patterns': ['Thanks', 'Thank you', "That's helpful"],
   'responses': ['Happy to help!', 'Any time!', 'My pleasure']},
  {'tag': 'chatbot',
   'patterns': ['Who built this chatbot?',
    'Tell me about Chatbot',
    'What is this chatbot name?'],
   'responses': ['Hi, I am Chatbot designed by Mayank.',
    'Thanks for asking. I am designed by Mayank Bajaj.',
    'I am a chatbot.']},
  {'tag': 'location',
   'patterns': ['What is your location?',
    'Where are you located?',
    'What is your address?'],
   'responses': ["We are from Worl

In [59]:
response('Suggest me some movies')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Lage Raho Munna Bhai


In [ ]:
response('Where are you located')

In [21]:
words=[]
classes=[]
documents=[]
ignore=['?']
#loop through each senetence in the intents pattern
for intent in intents['intents']:
  for pattern in intent['patterns']:
    #tokenize each and every word in the sentence
    w=nltk.word_tokenize(pattern)
    #add word to the word list
    words.extend(w)
    #add words todocuments
    documents.append((w,intent['tag']))
    #add tags to our classes list
    if intent['tag'] not in classes:
      classes.append(intent['tag'])


In [27]:
words=[]
classes=[]
documents=[]
ignore=['?']
#loop through each senetence in the intents pattern
for intent in intents['intents']:
  for pattern in intent['patterns']:
    #tokenize each and every word in the sentence
    w=nltk.word_tokenize(pattern)
    #add word to the word list
    words.extend(w)
    #add words todocuments
    documents.append((w,intent['tag']))
    #add tags to our classes list
    if intent['tag'] not in classes:
      classes.append(intent['tag'])

In [24]:
#perform steming and lower each word as well as remove duplicate
words=[stemmer.stem(w.lower()) for w in words if w not in ignore]
words=sorted(list(set(words)))

#remove duplicates classes

classes=sorted(list(set(classes)))
print(len(documents),"documents")
print(len(classes),"classes",classes)
print(len(words),"unique stemmed words",words)


27 documents
8 classes ['about', 'chatbot', 'connect', 'goodbye', 'greeting', 'location', 'movies', 'thanks']
52 unique stemmed words ["'s", 'about', 'account', 'address', 'ani', 'anyon', 'are', 'built', 'bye', 'can', 'chatbot', 'connect', 'day', 'favourit', 'give', 'good', 'goodby', 'hello', 'help', 'hi', 'how', 'i', 'is', 'later', 'link', 'locat', 'me', 'media', 'movi', 'name', 'out', 'reach', 'recommend', 'see', 'social', 'some', 'suggest', 'tell', 'thank', 'that', 'there', 'thi', 'to', 'way', 'we', 'what', 'where', 'which', 'who', 'you', 'your', 'yourself']


In [28]:
#create training data
train_x = []
train_y = []
#create an empty array for output
output_empty = [0] * len(classes)

#training set, bag of words for each sentence
for doc in documents:
  bag = []
  #list of tokenized words for pattern
  pattern_words = doc[0]
  #stem each word
  pattern_words = [stemmer.stem(word.lower()) for word in pattern_words]
  #create bag of words array
  for w in words:
    bag.append(1) if w in pattern_words else bag.append(0)

  #output is 1 for current tag and 0 for rest of other tag
  output_row = list(output_empty)
  output_row[classes.index(doc[1])] = 1

  # Append directly to train_x and train_y
  train_x.append(bag)
  train_y.append(output_row)

# Shuffle the data consistently
combined = list(zip(train_x, train_y))
random.shuffle(combined)
train_x_shuffled, train_y_shuffled = zip(*combined)

# Convert to NumPy arrays
train_x = np.array(train_x_shuffled)
train_y = np.array(train_y_shuffled)

In [29]:
# Build the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, input_shape=(len(train_x[0]),), activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(len(train_y[0]), activation='softmax')
])

# Compile the model
sgd = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

# Train the model
hist = model.fit(np.array(train_x), np.array(train_y), epochs=200, batch_size=5, verbose=1)

# Save the model
model.save('chatbot_model.h5')
print("Model created and saved.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0921 - loss: 2.1763
Epoch 2/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1571 - loss: 2.0814
Epoch 3/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2304 - loss: 2.0268    
Epoch 4/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3518 - loss: 1.7442
Epoch 5/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3549 - loss: 1.8002
Epoch 6/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4602 - loss: 1.6756 
Epoch 7/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4034 - loss: 1.9829 
Epoch 8/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3461 - loss: 1.7264 
Epoch 9/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3344 - loss: 1.6535 
Epoch 10/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4351 - loss: 1.7374 
Epoch 11/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5085 - loss: 1.4006 
Epoch 12/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4748 -

Model created and saved.


In [42]:
import pickle
pickle.dump( {'words':words, 'classes':classes, 'train_x':train_x, 'train_y':train_y}, open('training_data.pkl', 'wb') )

In [43]:
from keras.models import load_model
model = load_model('chatbot_model.h5')

In [34]:
#restoring all the data structures
data=pickle.load(open("training_data.pkl","rb"))
words=data['words']
classes=data['classes']


In [36]:
with open('intents.json') as json_data:
  intents=json.load(json_data)
#

In [37]:
def clean_up_sequence(sentence):
  #tokenize the pattern
  sentence_words=nltk.word_tokenize(sentence)
  sentence_words=[stemmer.stem(word.lower()) for word in sentence_words]
  return sentence_words

#returning bag of words array:0 or 1 for each word in the bag that exitsts in that sentence
def bow(setence,words):
  sentence_words=clean_up_sequence(setence)
  bag=[0]*len(words)
  for s in sentence_words:
    for i,w in enumerate(words):
      if w==s:
        bag[i]=1
  return(np.array(bag))

In [44]:
error_threshold=0.70
def classify(sentence):
  bag=bow(sentence,words)
  results=model.predict(np.array([bag]))[0]
  results=[[i,r] for i,r in enumerate(results) if r>error_threshold]
  results.sort(key=lambda x:x[1],reverse=True)
  return_list=[]
  for r in results:
    return_list.append({'intent':classes[r[0]],'probability':str(r[1])})
  return return_list

def response(sentence):
  results=classify(sentence)
  if results:
    while results:
      for i in intents['intents']:
        if i['tag']==results[0]['intent']:
          return print(random.choice(i['responses']))
      results.pop(0)

In [60]:
response('What is your location?')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Thans for your Interest. I live in India.
